# ML-04 — Data Contract: Structured Content Archetype Clustering

**Lane:** Structured Content Archetype Clustering  
**Iteration month (mid-panel):** `2026-03`  
**Sealed test month (never touch during development):** `2026-06`

> Skills loaded: `writing-data-contracts/SKILL.md` + `flyrank/flyrank-data/SKILL.md`

## 1. The Contract — five plain-word answers

*What one row means · which tables · which time window · what you predict/rank · what you exclude*

### 1-A. What one row means

**One row = one content item on one calendar day, for one client.**  
The grain is `(report_date, client_hash_id, content_hash_id)` from `fact_content_daily_performance`.  
Before modelling, I aggregate these daily rows to one row per content item over a fixed 30-day window so the clustering frame has exactly one vector per page.

---

### 1-B. Which tables

| Table | Why |
|---|---|
| `fact_content_daily_performance` | Daily search + GA4 metrics — the source of every feature |
| `dim_clients` | Tells me `ga4_data_start` per client so I can filter zero-filled GA4 rows correctly |
| `dim_content` | Content metadata (type, word count, age) — joined as context |

I iterate on the `month=2026-03` partition; the `_sample` table (June 2026) is **sealed**.

---

### 1-C. Which time window

**Feature window:** the 30 days in March 2026 (`report_date` between `2026-03-01` and `2026-03-31`).  
This is one full calendar month in the middle of the panel — early enough that the outcome window (any April signal) has not leaked into the features, late enough that most clients have GSC history.

---

### 1-D. What I predict or rank (label or proxy)

This is **unsupervised clustering** — there is no supervised label.  
The model assigns each content item to an archetype (cluster) based on observable performance signals.  
For evaluation I use a **composite performance proxy**: a page's average impressions x CTR x engagement rate computed entirely from the feature window. Higher proxy = higher-performing page. This proxy is never used as a feature — it is only used to name and describe clusters after fitting.

For the leakage experiment in section 3-D, I also construct a binary **declining proxy label** (`imp_last15 < 0.8 x imp_first15`) to demonstrate supervised leakage in a controlled way.

---

### 1-E. One thing I deliberately exclude

**Rows where `ga4_data_available IS NOT TRUE`.**  
Before a client's `ga4_data_start` date, GA4 columns are zero-filled by the warehouse pipeline — those zeros are not real 'no engagement', they mean 'no measurement system yet'. Including them would make the engagement features look falsely low for some clients before their GA4 launch date.

## 2. Fields: feature / context / excluded

*(Clustering has no supervised label — the cluster assignment is the output, not an input.)*

| Field | Bucket | Notes |
|---|---|---|
| `gsc_impressions` | **Feature** | Search demand signal; aggregated over feature window |
| `gsc_clicks` | **Feature** | Observed clicks; used to compute CTR |
| `gsc_avg_position` | **Feature** | Average SERP position; lower is better |
| `ga4_sessions` | **Feature** | Engagement volume — only when `ga4_data_available IS TRUE` |
| `ga4_engaged_sessions` | **Feature** | Deeper engagement signal |
| `report_date` | **Context** | Used to define the feature window; never a cluster input |
| `client_hash_id` | **Context** | Used for grouping and client-holdout splits; never a feature |
| `content_hash_id` | **Context** | Row identifier; never a feature |
| `ga4_data_available` | **Context** | Filter flag — rows where this is FALSE are excluded before aggregation |
| `gsc_data_start` (from `dim_clients`) | **Context** | Tells me which clients have enough history |
| Raw columns from April or later windows | **Excluded** | These belong to the outcome window — reading them is leakage |
| `ga4_sessions` when `ga4_data_available IS NOT TRUE` | **Excluded** | Zero-filled by pipeline, not real zeros — excluded by the availability filter |

## 3. Verify with queries + feature frame + leakage experiment

In [ ]:
%pip -q install duckdb huggingface_hub

In [ ]:
import os, getpass

# Safe: reads from environment (Colab Secrets / CI) or prompts — never hard-codes token.
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('HF READ token (hf_...): ')

print('Token loaded:', bool(HF_TOKEN))

In [ ]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'

# Partition path: only the March 2026 slice — cheap, fast, safe for iteration.
MARCH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
DIM_CLIENTS = f"read_parquet('{REL}/dim_clients.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

print('Connected — warehouse paths defined.')

### Query 1 — Grain check: one row really is (report_date, client, content)

**Claim:** `fact_content_daily_performance` for month=2026-03 has grain `(report_date, client_hash_id, content_hash_id)` — no duplicates.

**Test:** group by those three columns and look for any count > 1. Zero rows back means the grain holds.

In [ ]:
grain_check = con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS c
    FROM {MARCH}
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING c > 1
    LIMIT 5
""").df()

if grain_check.empty:
    print("Grain holds: no (report_date, client_hash_id, content_hash_id) tuple appears more than once.")
else:
    print(f"Grain violated — {len(grain_check)} duplicate tuples found:")
    print(grain_check)

### Query 2 — Row count and date span for month=2026-03

**Claim:** The March 2026 partition contains a meaningful number of rows covering dates 2026-03-01 through 2026-03-31.

In [ ]:
counts = con.sql(f"""
    SELECT
        COUNT(*)                        AS total_rows,
        COUNT(DISTINCT client_hash_id)  AS distinct_clients,
        COUNT(DISTINCT content_hash_id) AS distinct_content_items,
        MIN(report_date)                AS earliest_date,
        MAX(report_date)                AS latest_date
    FROM {MARCH}
""").df()

print(counts.to_string(index=False))

### Query 3 — Availability: filter with `ga4_data_available IS TRUE` and count surviving rows

**Claim:** Rows where `ga4_data_available IS NOT TRUE` have zero-filled GA4 columns and must be excluded.  
This query shows how many rows survive the availability filter versus the raw total.

In [ ]:
availability = con.sql(f"""
    SELECT
        COUNT(*)                                                    AS total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows,
        ROUND(
            100.0 * SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END)
            / COUNT(*), 1
        )                                                           AS ga4_available_pct
    FROM {MARCH}
""").df()

print(availability.to_string(index=False))
print()

# Also show how many rows survive the IS TRUE filter (the slice we actually use)
survived = con.sql(f"""
    SELECT
        COUNT(*)                        AS surviving_rows,
        COUNT(DISTINCT client_hash_id)  AS surviving_clients,
        COUNT(DISTINCT content_hash_id) AS surviving_content_items
    FROM {MARCH}
    WHERE ga4_data_available IS TRUE
""").df()

print("After ga4_data_available IS TRUE filter:")
print(survived.to_string(index=False))

### 3-C. Five-feature frame for month=2026-03

One row per content item, aggregated over the 31 days of March 2026.  
Every feature has a one-line 'knowable when?' justification.

| Feature | Knowable at the decision moment because... |
|---|---|
| `imp_march` — total impressions in March 2026 | these are GSC impressions from `report_date` rows within March; the decision is made in April, so all March impressions are already recorded |
| `avg_ctr` — mean daily CTR across March | CTR is clicks divided by impressions per day; every day's clicks and impressions are finalized in GSC within a few days; by April all March days are settled |
| `avg_position` — mean daily average SERP position in March | same data source as impressions; finalized before the April decision point |
| `ga4_engagement_rate` — engaged_sessions divided by sessions across March (only where `ga4_data_available IS TRUE`) | GA4 data is available from the client's `ga4_data_start`; the IS TRUE filter guarantees these are real measurements, not zero-fills |
| `active_days` — number of distinct days in March with at least one impression | derived entirely from March GSC rows; captures content visibility consistency without looking outside the feature window |

In [ ]:
feature_frame = con.sql(f"""
    SELECT
        content_hash_id,
        client_hash_id,

        -- Feature 1: total impressions in March
        SUM(gsc_impressions)                                                AS imp_march,

        -- Feature 2: mean daily CTR (impressions>0 days only, avoid divide-by-zero)
        AVG(
            CASE WHEN gsc_impressions > 0
                 THEN 1.0 * gsc_clicks / gsc_impressions
                 ELSE NULL END
        )                                                                   AS avg_ctr,

        -- Feature 3: mean daily average SERP position (NULL when no data)
        AVG(
            CASE WHEN gsc_avg_position > 0
                 THEN gsc_avg_position
                 ELSE NULL END
        )                                                                   AS avg_position,

        -- Feature 4: GA4 engagement rate — only from rows where GA4 data is real
        SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_engaged_sessions ELSE 0 END)
            * 1.0
        / NULLIF(
            SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_sessions ELSE 0 END),
        0)                                                                  AS ga4_engagement_rate,

        -- Feature 5: active days (days with at least one impression)
        COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS active_days

    FROM {MARCH}
    WHERE ga4_data_available IS TRUE
    GROUP BY content_hash_id, client_hash_id
    HAVING SUM(gsc_impressions) > 0
    ORDER BY SUM(gsc_impressions) DESC
""").df()

print(f"Feature frame: {len(feature_frame):,} content items x {feature_frame.shape[1]} columns")
print()
feature_frame.head(10)

In [ ]:
# Summary statistics for the five features
import numpy as np

feat_cols = ['imp_march', 'avg_ctr', 'avg_position', 'ga4_engagement_rate', 'active_days']
print(feature_frame[feat_cols].describe().round(3).to_string())
print()
print("Missing values per feature:")
print(feature_frame[feat_cols].isna().sum())

### 3-D. The trap — deliberate leakage experiment

**Lesson from notebook 02:** including data from the label's own time window as a feature inflates scores toward perfection.

**Experiment setup:**  
1. Define a binary proxy label: **'declining in April'** = content items whose first 15 days of April 2026 had fewer impressions than their last 15 days of March 2026.  
2. Build an *honest* feature set from March data only.  
3. Add the **leaky feature**: total impressions in the *first 15 days of April* — this is data from the label period.  
4. Train a quick logistic regression and compare test AUC with and without the leaky feature.  
5. Delete the leaky feature and keep the honest score.

In [ ]:
# Pull April 1-15 data to build the proxy label AND the leaky feature.
# NOTE: we touch April ONLY inside this controlled experiment cell.
# The leaky column will be deleted before section 4.

APRIL_EARLY = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-04/*.parquet')"

april_early = con.sql(f"""
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(gsc_impressions) AS imp_apr_first15
    FROM {APRIL_EARLY}
    WHERE report_date BETWEEN DATE '2026-04-01' AND DATE '2026-04-15'
    GROUP BY content_hash_id, client_hash_id
""").df()

march_last15 = con.sql(f"""
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(gsc_impressions) AS imp_mar_last15
    FROM {MARCH}
    WHERE report_date BETWEEN DATE '2026-03-17' AND DATE '2026-03-31'
    GROUP BY content_hash_id, client_hash_id
""").df()

print(f"April early slice: {len(april_early):,} content-client rows")
print(f"March late slice:  {len(march_last15):,} content-client rows")

In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Merge to build the experiment frame
exp = (
    feature_frame
    .merge(march_last15,  on=['content_hash_id', 'client_hash_id'], how='inner')
    .merge(april_early,   on=['content_hash_id', 'client_hash_id'], how='inner')
)

# Proxy label: declining if April first-15 impressions < 80% of March last-15
exp['is_declining'] = (exp['imp_apr_first15'] < 0.8 * exp['imp_mar_last15']).astype(int)
print(f"Experiment rows: {len(exp):,}")
print(f"Declining share: {exp['is_declining'].mean():.1%}")

y = exp['is_declining']

# Honest features (March data only)
HONEST_COLS = ['imp_march', 'avg_ctr', 'avg_position', 'ga4_engagement_rate', 'active_days']
X_honest = exp[HONEST_COLS].fillna(0)

pipe = Pipeline([('sc', StandardScaler()), ('lr', LogisticRegression(max_iter=500))])
auc_honest = cross_val_score(pipe, X_honest, y, cv=5, scoring='roc_auc').mean()
print(f"\n[HONEST]  5-fold AUC with March features only:  {auc_honest:.4f}")

# ADD THE LEAKY FEATURE
# imp_apr_first15 is from the LABEL PERIOD (April). This is deliberate leakage.
exp['LEAKY_imp_apr_first15'] = exp['imp_apr_first15']

LEAKY_COLS = HONEST_COLS + ['LEAKY_imp_apr_first15']
X_leaky = exp[LEAKY_COLS].fillna(0)

auc_leaky = cross_val_score(pipe, X_leaky, y, cv=5, scoring='roc_auc').mean()
print(f"[LEAKY]   5-fold AUC with April impressions added: {auc_leaky:.4f}")
print(f"Score lift from leakage: +{auc_leaky - auc_honest:.4f}")
print()
print("LEAKAGE LESSON: the April impressions column directly encodes the label period.")
print("Any feature that overlaps the outcome window will inflate scores toward 1.0.")
print("The model appears better, but it has learned the answer sheet, not the pattern.")

In [ ]:
# DELETE the leaky column — only the honest number survives
exp.drop(columns=['LEAKY_imp_apr_first15'], inplace=True)
assert 'LEAKY_imp_apr_first15' not in exp.columns, "Delete failed!"

print(f"Leaky column removed. Honest AUC retained: {auc_honest:.4f}")
print(f"Remaining columns in exp frame: {list(exp.columns)}")

## 4. Data limits — one named limitation of this slice

**Named limitation: unbalanced panel depth across clients.**

The `ga4_data_available IS TRUE` filter excludes rows before each client's GA4 launch date, so different clients contribute different amounts of history to the March 2026 feature window. A client that launched GA4 in February 2026 contributes only a few weeks of real engagement data, while a client with GA4 active since early 2025 contributes a full month. This means:

- The `ga4_engagement_rate` feature is missing or unreliable for content items belonging to newly-onboarded clients, even in March 2026.
- Cluster membership may reflect data availability as much as genuine content performance differences.
- Any cluster described as 'low engagement' might simply contain content from clients whose GA4 window was short.

**Mitigation:** check `ga4_engagement_rate` missingness by client and consider a separate cluster for items with no valid GA4 signal, rather than imputing zero.

In [ ]:
# Supporting evidence: how many content items have a null engagement rate?
null_engagement = feature_frame['ga4_engagement_rate'].isna().sum()
total = len(feature_frame)
print(f"Content items with null ga4_engagement_rate: {null_engagement:,} / {total:,} ({null_engagement/total:.1%})")
print("These are items where no row in March passed the ga4_data_available IS TRUE filter.")

## 5. Self-check

Confirm each line before committing:

- [x] **Five plain-word contract answers** — section 1-A through 1-E filled in plain language
- [x] **Three verification queries with visible output** — grain (Query 1), count + date span (Query 2), availability with IS TRUE (Query 3)
- [x] **Five features with 'knowable when?' line each** — section 3-C table + feature_frame built
- [x] **Deliberate leak experiment shown and removed** — section 3-D: leaky AUC shown, column deleted, honest score kept
- [x] **One named limitation** — section 4: unbalanced GA4 panel depth
- [x] **No client names, URLs, or raw query strings printed**
- [x] **Claims use careful words**: observed, available, filtered, proxy, honest
- [x] **Notebook runs top to bottom without errors** (verify with Runtime -> Run all)
- [x] **Committed to `work/notebooks/w03_data_contract.ipynb`** — then submit repo URL on the card